In [1]:
import sys
sys.path.insert(0, '/home/jy/tza-pypsa')

# Now your imports will use the local version
import pypsa
from tz_pypsa.constraints import (
    constr_max_annual_utilisation_generator, 
    constr_min_annual_utilisation_generator,
    constr_max_annual_utilisation_links,
    constr_min_annual_utilisation_links,
    constr_max_annual_utilisation_storage_discharge, 
    constr_min_annual_utilisation_storage_discharge, 
    constr_max_annual_utilisation_storage_charge,     
    constr_min_annual_utilisation_storage_charge,     
    constr_soc_intraday_profile,
    constr_soc_weekly_profile,
    constr_production_target_max,
    constr_production_target_min,
    constr_max_ramps_daily
)

import plotly.express as px
import pandas as pd     
import numpy as np
import xarray as xr
import os
os.environ['GRB_LICENSE_FILE'] = '/home/jy/opt/gurobi/gurobi.lic'

In [2]:
n = pypsa.Network()
# n.import_from_netcdf("/home/jy/Backup/client-earth_CE-A-OCCTO-003_v3/platform_network.nc") # calibration
n.import_from_netcdf("/home/jy/tz-amp/.amp/client-earth_CE-A-SEP7-005_v3/platform_network.nc")

INFO:pypsa.io:Imported network platform_network.nc has buses, carriers, generators, links, loads, storage_units


In [3]:
n.generators_t.marginal_cost.loc[:, n.generators_t.marginal_cost.filter(like="coal").columns] = n.generators_t.marginal_cost.loc[:, n.generators_t.marginal_cost.filter(like="coal-unspecified").columns].max().max()
n.generators_t.marginal_cost.loc[:, n.generators_t.marginal_cost.filter(like="gas").columns] = n.generators_t.marginal_cost.loc[:, n.generators_t.marginal_cost.filter(like="gas-unspecified").columns].max().max()

In [ ]:
n_solved = pypsa.Network()
n_solved.import_from_netcdf("/home/jy/tz-amp/.amp/client-earth_CE-A-SEP7-005_v3/platform_network.solved.nc")

In [ ]:
n.generators.loc[n_solved.generators.p_nom_extendable, "p_nom"] = np.ceil(n_solved.generators.loc[n_solved.generators.p_nom_extendable, "p_nom_opt"])
n.storage_units.loc[n_solved.storage_units.p_nom_extendable, "p_nom"] = np.ceil(n_solved.storage_units.loc[n_solved.storage_units.p_nom_extendable, "p_nom_opt"])

In [4]:
n.generators['carrier'] = n.generators['type']
n.links['carrier'] = n.links['type']
n.storage_units['carrier'] = n.storage_units['type']

In [5]:
all_carriers = (
    n.generators.carrier.unique().tolist()
    + n.storage_units.carrier.unique().tolist()
    + n.links.carrier.unique().tolist()
)
missing_carriers = set(all_carriers) - set(n.carriers.index)
if missing_carriers:
    n.add("Carrier", missing_carriers)

n.generators.build_year = 2040
n.storage_units.build_year = 2040
n.links.build_year = 2040

In [6]:
n.generators.loc[n.generators.carrier == 'wind-offshore-unspecified', 'p_nom_extendable'] = True
n.generators.loc[n.generators.carrier == 'wind-onshore', 'p_nom_extendable'] = True
n.generators.loc[n.generators.carrier == 'photovoltaic-unspecified', 'p_nom_extendable'] = True
n.generators.loc[n.generators.carrier == 'geothermal-unspecified', 'p_nom_extendable'] = True

n.generators.loc[n.generators.carrier == 'gas-ccs', 'p_nom_extendable'] = True
n.generators.loc[n.generators.carrier == 'coal-ammonia-cofiring', 'p_nom_extendable'] = True
n.generators.loc[n.generators.carrier == 'gas-hydrogen-cofiring', 'p_nom_extendable'] = True
n.storage_units.loc[n.storage_units.carrier == 'utility-scale', 'p_nom_extendable'] = True

In [7]:
# set p_nom_min = p_nom to avoid capacity retirement
n.generators.p_nom_min = n.generators.p_nom

In [8]:
# Renewable p_nom_max
# p_nom_max for geothermal
n.generators.loc[n.generators.carrier == 'geothermal-unspecified', 'p_nom_max'] = n.generators.loc[n.generators.carrier == 'geothermal-unspecified', 'p_nom'] * 1.5

# p_nom_max for solar
n.generators.loc[(n.generators.carrier == 'photovoltaic-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-HK'), 'p_nom_max'] = 8305 
n.generators.loc[(n.generators.carrier == 'photovoltaic-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-TH'), 'p_nom_max'] = 33780 
n.generators.loc[(n.generators.carrier == 'photovoltaic-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-TK'), 'p_nom_max'] = 60231 
n.generators.loc[(n.generators.carrier == 'photovoltaic-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-CB'), 'p_nom_max'] = 38999 
n.generators.loc[(n.generators.carrier == 'photovoltaic-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-HR'), 'p_nom_max'] = 4970 
n.generators.loc[(n.generators.carrier == 'photovoltaic-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-KA'), 'p_nom_max'] = 23097 
n.generators.loc[(n.generators.carrier == 'photovoltaic-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-CG'), 'p_nom_max'] = 26285 
n.generators.loc[(n.generators.carrier == 'photovoltaic-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-SH'), 'p_nom_max'] = 13496 
n.generators.loc[(n.generators.carrier == 'photovoltaic-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-KY'), 'p_nom_max'] = 49404

# p_nom_max for onshore wind
n.generators.loc[(n.generators.carrier == 'wind-onshore') & (n.generators.bus == 'GRIDREGION-JPN-HK'), 'p_nom_max'] = 6290
n.generators.loc[(n.generators.carrier == 'wind-onshore') & (n.generators.bus == 'GRIDREGION-JPN-TH'), 'p_nom_max'] = 18246
n.generators.loc[(n.generators.carrier == 'wind-onshore') & (n.generators.bus == 'GRIDREGION-JPN-TK'), 'p_nom_max'] = 3890
n.generators.loc[(n.generators.carrier == 'wind-onshore') & (n.generators.bus == 'GRIDREGION-JPN-CB'), 'p_nom_max'] = 1221
n.generators.loc[(n.generators.carrier == 'wind-onshore') & (n.generators.bus == 'GRIDREGION-JPN-HR'), 'p_nom_max'] = 1789
n.generators.loc[(n.generators.carrier == 'wind-onshore') & (n.generators.bus == 'GRIDREGION-JPN-KA'), 'p_nom_max'] = 2404
n.generators.loc[(n.generators.carrier == 'wind-onshore') & (n.generators.bus == 'GRIDREGION-JPN-CG'), 'p_nom_max'] = 2144
n.generators.loc[(n.generators.carrier == 'wind-onshore') & (n.generators.bus == 'GRIDREGION-JPN-SH'), 'p_nom_max'] = 1950
n.generators.loc[(n.generators.carrier == 'wind-onshore') & (n.generators.bus == 'GRIDREGION-JPN-KY'), 'p_nom_max'] = 3039

# p_nom_max for offshore wind
# Taking highest quality sites for each region
n.generators.loc[(n.generators.carrier == 'wind-offshore-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-HK'), 'p_nom_max'] = 45106
n.generators.loc[(n.generators.carrier == 'wind-offshore-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-TH'), 'p_nom_max'] = 24536
n.generators.loc[(n.generators.carrier == 'wind-offshore-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-TK'), 'p_nom_max'] = 16176
n.generators.loc[(n.generators.carrier == 'wind-offshore-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-CB'), 'p_nom_max'] = 10378
n.generators.loc[(n.generators.carrier == 'wind-offshore-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-HR'), 'p_nom_max'] = 1300
n.generators.loc[(n.generators.carrier == 'wind-offshore-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-KA'), 'p_nom_max'] = 1150
n.generators.loc[(n.generators.carrier == 'wind-offshore-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-CG'), 'p_nom_max'] = 843
n.generators.loc[(n.generators.carrier == 'wind-offshore-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-SH'), 'p_nom_max'] = 1982 
n.generators.loc[(n.generators.carrier == 'wind-offshore-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-KY'), 'p_nom_max'] = 13141

# # p_nom_max for batteries
# n.storage_units.loc[(n.storage_units.carrier == 'utility-scale') & (n.storage_units.bus == 'GRIDREGION-JPN-HK'), 'p_nom_max'] = 610
# n.storage_units.loc[(n.storage_units.carrier == 'utility-scale') & (n.storage_units.bus == 'GRIDREGION-JPN-TH'), 'p_nom_max'] = 1711
# n.storage_units.loc[(n.storage_units.carrier == 'utility-scale') & (n.storage_units.bus == 'GRIDREGION-JPN-TK'), 'p_nom_max'] = 6549
# n.storage_units.loc[(n.storage_units.carrier == 'utility-scale') & (n.storage_units.bus == 'GRIDREGION-JPN-CB'), 'p_nom_max'] = 3034
# n.storage_units.loc[(n.storage_units.carrier == 'utility-scale') & (n.storage_units.bus == 'GRIDREGION-JPN-HR'), 'p_nom_max'] = 616
# n.storage_units.loc[(n.storage_units.carrier == 'utility-scale') & (n.storage_units.bus == 'GRIDREGION-JPN-KA'), 'p_nom_max'] = 3327
# n.storage_units.loc[(n.storage_units.carrier == 'utility-scale') & (n.storage_units.bus == 'GRIDREGION-JPN-CG'), 'p_nom_max'] = 1273
# n.storage_units.loc[(n.storage_units.carrier == 'utility-scale') & (n.storage_units.bus == 'GRIDREGION-JPN-SH'), 'p_nom_max'] = 592
# n.storage_units.loc[(n.storage_units.carrier == 'utility-scale') & (n.storage_units.bus == 'GRIDREGION-JPN-KY'), 'p_nom_max'] = 1859

In [9]:
# p_nom_max for coal-ammonia-cofiring
n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-HK'), 'p_nom_max'] = 280 + n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-HK'), 'p_nom_min']
n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-TH'), 'p_nom_max'] = 1035 + n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-TH'), 'p_nom_min']
n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-TK'), 'p_nom_max'] = 1166 + n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-TK'), 'p_nom_min']
n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-CB'), 'p_nom_max'] = 661 + n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-CB'), 'p_nom_min']
n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-HR'), 'p_nom_max'] = 344 + n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-HR'), 'p_nom_min']
n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-KA'), 'p_nom_max'] = 670 + n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-KA'), 'p_nom_min']
n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-CG'), 'p_nom_max'] = 824 + n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-CG'), 'p_nom_min']
n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-SH'), 'p_nom_max'] = 573 + n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-SH'), 'p_nom_min']
n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-KY'), 'p_nom_max'] = 902 + n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-KY'), 'p_nom_min']

In [10]:
# p_nom_max for gas-hydrogen-cofiring
n.generators.loc[(n.generators.carrier == 'gas-hydrogen-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-HK'), 'p_nom_max'] = 105 + n.generators.loc[(n.generators.carrier == 'gas-hydrogen-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-HK'), 'p_nom_min']
n.generators.loc[(n.generators.carrier == 'gas-hydrogen-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-TH'), 'p_nom_max'] = 652 + n.generators.loc[(n.generators.carrier == 'gas-hydrogen-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-TH'), 'p_nom_min']
n.generators.loc[(n.generators.carrier == 'gas-hydrogen-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-TK'), 'p_nom_max'] = 2585 + n.generators.loc[(n.generators.carrier == 'gas-hydrogen-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-TK'), 'p_nom_min']
n.generators.loc[(n.generators.carrier == 'gas-hydrogen-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-CB'), 'p_nom_max'] = 1200 + n.generators.loc[(n.generators.carrier == 'gas-hydrogen-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-CB'), 'p_nom_min']
n.generators.loc[(n.generators.carrier == 'gas-hydrogen-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-HR'), 'p_nom_max'] = 74 + n.generators.loc[(n.generators.carrier == 'gas-hydrogen-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-HR'), 'p_nom_min']
n.generators.loc[(n.generators.carrier == 'gas-hydrogen-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-KA'), 'p_nom_max'] = 893 + n.generators.loc[(n.generators.carrier == 'gas-hydrogen-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-KA'), 'p_nom_min']
n.generators.loc[(n.generators.carrier == 'gas-hydrogen-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-CG'), 'p_nom_max'] = 190 + n.generators.loc[(n.generators.carrier == 'gas-hydrogen-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-CG'), 'p_nom_min']
n.generators.loc[(n.generators.carrier == 'gas-hydrogen-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-SH'), 'p_nom_max'] = 85 + n.generators.loc[(n.generators.carrier == 'gas-hydrogen-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-SH'), 'p_nom_min']
n.generators.loc[(n.generators.carrier == 'gas-hydrogen-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-KY'), 'p_nom_max'] = 402 + n.generators.loc[(n.generators.carrier == 'gas-hydrogen-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-KY'), 'p_nom_min']

In [ ]:
n.generators.groupby('type')[['p_nom', 'p_nom_max']].sum()

In [11]:
n.generators.loc[(n.generators.carrier == 'coal-unspecified'), 'ramp_limit_up'] = 0.4
n.generators.loc[(n.generators.carrier == 'coal-unspecified'), 'ramp_limit_down'] = 0.4

n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring'), 'ramp_limit_up'] = 0.4
n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring'), 'ramp_limit_down'] = 0.4

n.generators.loc[(n.generators.carrier == 'gas-unspecified'), 'ramp_limit_up'] = 0.9
n.generators.loc[(n.generators.carrier == 'gas-unspecified'), 'ramp_limit_down'] = 0.9

n.generators.loc[(n.generators.carrier == 'gas-hydrogen-cofiring'), 'ramp_limit_up'] = 0.9
n.generators.loc[(n.generators.carrier == 'gas-hydrogen-cofiring'), 'ramp_limit_down'] = 0.9

n.generators.loc[(n.generators.carrier == 'gas-ccs'), 'ramp_limit_up'] = 0.9
n.generators.loc[(n.generators.carrier == 'gas-ccs'), 'ramp_limit_down'] = 0.9

n.generators.loc[(n.generators.carrier == 'nuclear'), 'ramp_limit_up'] = 0.6
n.generators.loc[(n.generators.carrier == 'nuclear'), 'ramp_limit_down'] = 0.6

In [12]:
n.storage_units.loc[n.storage_units.carrier == 'utility-scale', 'efficiency_store'] = 0.92
n.storage_units.loc[n.storage_units.carrier == 'utility-scale', 'efficiency_dispatch'] = 0.92

In [ ]:
n.generators.loc[n.generators.carrier == 'wind-offshore-unspecified', 'marginal_cost'] = -1
n.generators.loc[n.generators.carrier == 'wind-onshore', 'marginal_cost'] = -1
n.generators.loc[n.generators.carrier == 'photovoltaic-unspecified', 'marginal_cost'] = -1

In [ ]:
n.generators.loc[n.generators.carrier == 'nuclear', 'max_ramps_per_day'] = 2

In [13]:
n.generators_t.p_min_pu = n.generators_t.p_max_pu.filter(regex='biomass|geothermal|hydro')

In [14]:
# p_max_pu - coal
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-HK'), 'p_max_pu'] = 0.46
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-TH'), 'p_max_pu'] = 0.57
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-TK'), 'p_max_pu'] = 0.62
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-CB'), 'p_max_pu'] = 0.63
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-HR'), 'p_max_pu'] = 0.60
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-KA'), 'p_max_pu'] = 0.60
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-CG'), 'p_max_pu'] = 0.52
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-SH'), 'p_max_pu'] = 0.59
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-KY'), 'p_max_pu'] = 0.51

# p_max_pu - coal-ammonia cofiring
n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-HK'), 'p_max_pu'] = 0.46
n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-TH'), 'p_max_pu'] = 0.57
n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-TK'), 'p_max_pu'] = 0.62
n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-CB'), 'p_max_pu'] = 0.63
n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-HR'), 'p_max_pu'] = 0.60
n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-KA'), 'p_max_pu'] = 0.60
n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-CG'), 'p_max_pu'] = 0.52
n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-SH'), 'p_max_pu'] = 0.59
n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-KY'), 'p_max_pu'] = 0.51

In [15]:
# max_utilisation_rate
# coal
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-HK'), 'max_utilisation_rate'] = 0.34
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-TH'), 'max_utilisation_rate'] = 0.445
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-TK'), 'max_utilisation_rate'] = 0.52
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-CB'), 'max_utilisation_rate'] = 0.55
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-HR'), 'max_utilisation_rate'] = 0.51
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-KA'), 'max_utilisation_rate'] = 0.52
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-CG'), 'max_utilisation_rate'] = 0.41
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-SH'), 'max_utilisation_rate'] = 0.45
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-KY'), 'max_utilisation_rate'] = 0.365

# coal-ammonia cofiring
n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-HK'), 'max_utilisation_rate'] = 0.34
n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-TH'), 'max_utilisation_rate'] = 0.445
n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-TK'), 'max_utilisation_rate'] = 0.52
n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-CB'), 'max_utilisation_rate'] = 0.55
n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-HR'), 'max_utilisation_rate'] = 0.51
n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-KA'), 'max_utilisation_rate'] = 0.52
n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-CG'), 'max_utilisation_rate'] = 0.41
n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-SH'), 'max_utilisation_rate'] = 0.45
n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-KY'), 'max_utilisation_rate'] = 0.365

# gas
n.generators.loc[(n.generators.carrier == 'gas-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-HK'), 'max_utilisation_rate'] = 0.70
n.generators.loc[(n.generators.carrier == 'gas-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-TH'), 'max_utilisation_rate'] = 0.70
n.generators.loc[(n.generators.carrier == 'gas-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-TK'), 'max_utilisation_rate'] = 0.70
n.generators.loc[(n.generators.carrier == 'gas-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-CB'), 'max_utilisation_rate'] = 0.70
n.generators.loc[(n.generators.carrier == 'gas-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-HR'), 'max_utilisation_rate'] = 0.70
n.generators.loc[(n.generators.carrier == 'gas-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-KA'), 'max_utilisation_rate'] = 0.70
n.generators.loc[(n.generators.carrier == 'gas-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-CG'), 'max_utilisation_rate'] = 0.70
n.generators.loc[(n.generators.carrier == 'gas-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-SH'), 'max_utilisation_rate'] = 0.70
n.generators.loc[(n.generators.carrier == 'gas-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-KY'), 'max_utilisation_rate'] = 0.70

# gas-ccs
n.generators.loc[(n.generators.carrier == 'gas-ccs') & (n.generators.bus == 'GRIDREGION-JPN-HK'), 'max_utilisation_rate'] = 0.70
n.generators.loc[(n.generators.carrier == 'gas-ccs') & (n.generators.bus == 'GRIDREGION-JPN-TH'), 'max_utilisation_rate'] = 0.70
n.generators.loc[(n.generators.carrier == 'gas-ccs') & (n.generators.bus == 'GRIDREGION-JPN-TK'), 'max_utilisation_rate'] = 0.70
n.generators.loc[(n.generators.carrier == 'gas-ccs') & (n.generators.bus == 'GRIDREGION-JPN-CB'), 'max_utilisation_rate'] = 0.70
n.generators.loc[(n.generators.carrier == 'gas-ccs') & (n.generators.bus == 'GRIDREGION-JPN-HR'), 'max_utilisation_rate'] = 0.70
n.generators.loc[(n.generators.carrier == 'gas-ccs') & (n.generators.bus == 'GRIDREGION-JPN-KA'), 'max_utilisation_rate'] = 0.70
n.generators.loc[(n.generators.carrier == 'gas-ccs') & (n.generators.bus == 'GRIDREGION-JPN-CG'), 'max_utilisation_rate'] = 0.70
n.generators.loc[(n.generators.carrier == 'gas-ccs') & (n.generators.bus == 'GRIDREGION-JPN-SH'), 'max_utilisation_rate'] = 0.70
n.generators.loc[(n.generators.carrier == 'gas-ccs') & (n.generators.bus == 'GRIDREGION-JPN-KY'), 'max_utilisation_rate'] = 0.70

# gas-hydrogen-cofiring
n.generators.loc[(n.generators.carrier == 'gas-hydrogen-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-HK'), 'max_utilisation_rate'] = 0.70
n.generators.loc[(n.generators.carrier == 'gas-hydrogen-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-TH'), 'max_utilisation_rate'] = 0.70
n.generators.loc[(n.generators.carrier == 'gas-hydrogen-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-TK'), 'max_utilisation_rate'] = 0.70
n.generators.loc[(n.generators.carrier == 'gas-hydrogen-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-CB'), 'max_utilisation_rate'] = 0.70
n.generators.loc[(n.generators.carrier == 'gas-hydrogen-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-HR'), 'max_utilisation_rate'] = 0.70
n.generators.loc[(n.generators.carrier == 'gas-hydrogen-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-KA'), 'max_utilisation_rate'] = 0.70
n.generators.loc[(n.generators.carrier == 'gas-hydrogen-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-CG'), 'max_utilisation_rate'] = 0.70
n.generators.loc[(n.generators.carrier == 'gas-hydrogen-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-SH'), 'max_utilisation_rate'] = 0.70
n.generators.loc[(n.generators.carrier == 'gas-hydrogen-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-KY'), 'max_utilisation_rate'] = 0.70


In [16]:
# min_utilisation_rate
# coal
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-HK'), 'min_utilisation_rate'] = 0.33
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-TH'), 'min_utilisation_rate'] = 0.44
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-TK'), 'min_utilisation_rate'] = 0.51
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-KY'), 'min_utilisation_rate'] = 0.36

# coal-ammonia-cofiring
n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-HK'), 'min_utilisation_rate'] = 0.33
n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-TH'), 'min_utilisation_rate'] = 0.44
n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-TK'), 'min_utilisation_rate'] = 0.51
n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-KY'), 'min_utilisation_rate'] = 0.36

# gas - calibration constraints
n.generators.loc[(n.generators.carrier == 'gas-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-HK'), 'min_utilisation_rate'] = 0.14
n.generators.loc[(n.generators.carrier == 'gas-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-TH'), 'min_utilisation_rate'] = 0.095
n.generators.loc[(n.generators.carrier == 'gas-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-TK'), 'min_utilisation_rate'] = 0.16
n.generators.loc[(n.generators.carrier == 'gas-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-CB'), 'min_utilisation_rate'] = 0.23
n.generators.loc[(n.generators.carrier == 'gas-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-HR'), 'min_utilisation_rate'] = 0.10
n.generators.loc[(n.generators.carrier == 'gas-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-KA'), 'min_utilisation_rate'] = 0.225
n.generators.loc[(n.generators.carrier == 'gas-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-CG'), 'min_utilisation_rate'] = 0.05
n.generators.loc[(n.generators.carrier == 'gas-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-SH'), 'min_utilisation_rate'] = 0.085
n.generators.loc[(n.generators.carrier == 'gas-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-KY'), 'min_utilisation_rate'] = 0.04

# Assume gas-hydrogen-cofiring to follow the same min utilisation rate as gas to reflect the same level of operational constraints
n.generators.loc[(n.generators.carrier == 'gas-hydrogen-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-HK'), 'min_utilisation_rate'] = 0.14
n.generators.loc[(n.generators.carrier == 'gas-hydrogen-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-TH'), 'min_utilisation_rate'] = 0.095
n.generators.loc[(n.generators.carrier == 'gas-hydrogen-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-TK'), 'min_utilisation_rate'] = 0.16
n.generators.loc[(n.generators.carrier == 'gas-hydrogen-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-CB'), 'min_utilisation_rate'] = 0.23
n.generators.loc[(n.generators.carrier == 'gas-hydrogen-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-HR'), 'min_utilisation_rate'] = 0.10
n.generators.loc[(n.generators.carrier == 'gas-hydrogen-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-KA'), 'min_utilisation_rate'] = 0.225
n.generators.loc[(n.generators.carrier == 'gas-hydrogen-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-CG'), 'min_utilisation_rate'] = 0.05
n.generators.loc[(n.generators.carrier == 'gas-hydrogen-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-SH'), 'min_utilisation_rate'] = 0.085
n.generators.loc[(n.generators.carrier == 'gas-hydrogen-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-KY'), 'min_utilisation_rate'] = 0.04

# Assume gas-ccs to follow the same min utilisation rate as gas to reflect the same level of operational constraints
n.generators.loc[(n.generators.carrier == 'gas-ccs') & (n.generators.bus == 'GRIDREGION-JPN-HK'), 'min_utilisation_rate'] = 0.14
n.generators.loc[(n.generators.carrier == 'gas-ccs') & (n.generators.bus == 'GRIDREGION-JPN-TH'), 'min_utilisation_rate'] = 0.095
n.generators.loc[(n.generators.carrier == 'gas-ccs') & (n.generators.bus == 'GRIDREGION-JPN-TK'), 'min_utilisation_rate'] = 0.16
n.generators.loc[(n.generators.carrier == 'gas-ccs') & (n.generators.bus == 'GRIDREGION-JPN-CB'), 'min_utilisation_rate'] = 0.23
n.generators.loc[(n.generators.carrier == 'gas-ccs') & (n.generators.bus == 'GRIDREGION-JPN-HR'), 'min_utilisation_rate'] = 0.10
n.generators.loc[(n.generators.carrier == 'gas-ccs') & (n.generators.bus == 'GRIDREGION-JPN-KA'), 'min_utilisation_rate'] = 0.225
n.generators.loc[(n.generators.carrier == 'gas-ccs') & (n.generators.bus == 'GRIDREGION-JPN-CG'), 'min_utilisation_rate'] = 0.05
n.generators.loc[(n.generators.carrier == 'gas-ccs') & (n.generators.bus == 'GRIDREGION-JPN-SH'), 'min_utilisation_rate'] = 0.085
n.generators.loc[(n.generators.carrier == 'gas-ccs') & (n.generators.bus == 'GRIDREGION-JPN-KY'), 'min_utilisation_rate'] = 0.04

In [17]:
# max_utilisation_rate - transmission
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-HK~GRIDREGION-JPN-TH'), 'max_utilisation_rate'] = 0.48 + 0.025
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-TH~GRIDREGION-JPN-HK'), 'max_utilisation_rate'] = 0.00 + 0.025
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-TH~GRIDREGION-JPN-TK'), 'max_utilisation_rate'] = 0.5875 + 0.025
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-TK~GRIDREGION-JPN-TH'), 'max_utilisation_rate'] = 0.00 + 0.025
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-TK~GRIDREGION-JPN-CB'), 'max_utilisation_rate'] = 0.44 + 0.025
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-CB~GRIDREGION-JPN-SH'), 'max_utilisation_rate'] = 0.00 + 0.025
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-CB~GRIDREGION-JPN-TK'), 'max_utilisation_rate'] = 0.25 + 0.025
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-CB~GRIDREGION-JPN-HR'), 'max_utilisation_rate'] = 0.02 + 0.025
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-CB~GRIDREGION-JPN-KA'), 'max_utilisation_rate'] = 0.04 + 0.025
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-HR~GRIDREGION-JPN-CB'), 'max_utilisation_rate'] = 0.33 + 0.025
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-HR~GRIDREGION-JPN-KA'), 'max_utilisation_rate'] = 0.18 + 0.025
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-KA~GRIDREGION-JPN-HR'), 'max_utilisation_rate'] = 0.09 + 0.025
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-KA~GRIDREGION-JPN-CB'), 'max_utilisation_rate'] = 0.25 + 0.025
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-KA~GRIDREGION-JPN-CG'), 'max_utilisation_rate'] = 0.19 + 0.025
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-KA~GRIDREGION-JPN-SH'), 'max_utilisation_rate'] = 0.00 + 0.025
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-CG~GRIDREGION-JPN-KA'), 'max_utilisation_rate'] = 0.40 + 0.025
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-CG~GRIDREGION-JPN-SH'), 'max_utilisation_rate'] = 0.00 + 0.025
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-SH~GRIDREGION-JPN-KY'), 'max_utilisation_rate'] = 0.00 + 0.025
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-KY~GRIDREGION-JPN-CG'), 'max_utilisation_rate'] = 0.34 + 0.025
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-KY~GRIDREGION-JPN-SH'), 'max_utilisation_rate'] = 0.83 + 0.025

In [18]:
# min_utilisation_rate - transmission
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-HK~GRIDREGION-JPN-TH'), 'min_utilisation_rate'] = 0.48 - 0.025
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-TH~GRIDREGION-JPN-TK'), 'min_utilisation_rate'] = 0.5875 - 0.025
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-TK~GRIDREGION-JPN-CB'), 'min_utilisation_rate'] = 0.44 - 0.025
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-CB~GRIDREGION-JPN-TK'), 'min_utilisation_rate'] = 0.25 - 0.025
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-CB~GRIDREGION-JPN-KA'), 'min_utilisation_rate'] = 0.04 - 0.025
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-HR~GRIDREGION-JPN-CB'), 'min_utilisation_rate'] = 0.32 - 0.025
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-KA~GRIDREGION-JPN-CB'), 'min_utilisation_rate'] = 0.25 - 0.025
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-KA~GRIDREGION-JPN-HR'), 'min_utilisation_rate'] = 0.09 - 0.025
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-KA~GRIDREGION-JPN-CG'), 'min_utilisation_rate'] = 0.19 - 0.025
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-CG~GRIDREGION-JPN-KA'), 'min_utilisation_rate'] = 0.40 - 0.025
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-CG~GRIDREGION-JPN-KY'), 'min_utilisation_rate'] = 0.19 - 0.025
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-SH~GRIDREGION-JPN-KA'), 'min_utilisation_rate'] = 0.98 - 0.025
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-SH~GRIDREGION-JPN-CG'), 'min_utilisation_rate'] = 0.91 - 0.025

In [19]:
# max_utilisation_rate
# hydro-pumped-storage
n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-HK'), 'discharge_min_utilisation_rate'] = 0.126
n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-HK'), 'charge_max_utilisation_rate'] = 0.180

n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-TH'), 'discharge_min_utilisation_rate'] = 0.136
n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-TH'), 'charge_max_utilisation_rate'] = 0.195

n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-TK'), 'discharge_min_utilisation_rate'] = 0.121
n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-TK'), 'charge_max_utilisation_rate'] = 0.174

n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-CB'), 'discharge_min_utilisation_rate'] = 0.059
n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-CB'), 'charge_max_utilisation_rate'] = 0.085

n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-HR'), 'discharge_min_utilisation_rate'] = 0.055
n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-HR'), 'charge_max_utilisation_rate'] = 0.080

n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-KA'), 'discharge_min_utilisation_rate'] = 0.056
n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-KA'), 'charge_max_utilisation_rate'] = 0.080

n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-CG'), 'discharge_min_utilisation_rate'] = 0.056
n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-CG'), 'charge_max_utilisation_rate'] = 0.081

n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-SH'), 'discharge_min_utilisation_rate'] = 0.064
n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-SH'), 'charge_max_utilisation_rate'] = 0.092

n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-KY'), 'discharge_min_utilisation_rate'] = 0.057
n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-KY'), 'charge_max_utilisation_rate'] = 0.083

In [20]:
n.optimize.create_model()
# constr_max_annual_utilisation_generator(n, carriers='coal-unspecified|gas-unspecified') # Calibration
# constr_min_annual_utilisation_generator(n, carriers='coal-unspecified|gas-unspecified') # Calibration
constr_max_annual_utilisation_generator(n, carriers='coal-unspecified|gas-unspecified|gas-hydrogen-cofiring|coal-ammonia-cofiring|gas-ccs') # Set max annual utilisation for these generators
constr_min_annual_utilisation_generator(n, carriers='coal-unspecified|gas-unspecified|gas-hydrogen-cofiring|coal-ammonia-cofiring|gas-ccs') # Set min annual utilisation for these generators
constr_max_annual_utilisation_links(n, carriers='transmission') # Set max annual utilisation for these links
constr_min_annual_utilisation_links(n, carriers='transmission') # Set min annual utilisation for these links
# constr_min_annual_utilisation_storage_discharge(n, carriers='hydro-pumped-storage-unspecified') # Set min annual utilisation for these storage units
# constr_max_annual_utilisation_storage_charge(n, carriers='hydro-pumped-storage-unspecified') # Set min annual utilisation for these storage units
# constr_soc_intraday_profile(
#     n, 
#     max_csv="/home/jy/Backup/client-earth_CE-A-OCCTO-003_testing/templates/state_of_charge_intraday_profile_annual_max.csv",
#     min_csv="/home/jy/Backup/client-earth_CE-A-OCCTO-003_testing/templates/state_of_charge_intraday_profile_annual_min.csv"
# )
# constr_soc_weekly_profile(
#     n, 
#     max_csv="/home/jy/Backup/client-earth_CE-A-OCCTO-003_testing/templates/state_of_charge_weekly_profile_annual_max.csv",
#     min_csv="/home/jy/Backup/client-earth_CE-A-OCCTO-003_testing/templates/state_of_charge_weekly_profile_annual_min.csv",
#     day_shift=2
# )
constr_production_target_min(n, 
                            ['GRIDREGION-JPN-SH', 'GRIDREGION-JPN-HR', 'GRIDREGION-JPN-CB', 
                            'GRIDREGION-JPN-HK', 'GRIDREGION-JPN-KA', 'GRIDREGION-JPN-CG', 
                            'GRIDREGION-JPN-TK', 'GRIDREGION-JPN-TH', 'GRIDREGION-JPN-KY'],
                            ['wind-offshore-unspecified','photovoltaic-unspecified', 'wind-onshore', 'geothermal-unspecified', 'biomass', 'hydro-reservoir-and-run-of-river'],
                            0.50)

constr_production_target_max(n, 
                            ['GRIDREGION-JPN-SH', 'GRIDREGION-JPN-HR', 'GRIDREGION-JPN-CB', 
                            'GRIDREGION-JPN-HK', 'GRIDREGION-JPN-KA', 'GRIDREGION-JPN-CG', 
                            'GRIDREGION-JPN-TK', 'GRIDREGION-JPN-TH', 'GRIDREGION-JPN-KY'],
                            ['wind-offshore-unspecified','photovoltaic-unspecified', 'wind-onshore', 'geothermal-unspecified', 'biomass', 'hydro-reservoir-and-run-of-river'],
                            0.51)

['gas-ccs:GRIDREGION-JPN-SH', 'gas-ccs:GRIDREGION-JPN-HR', 'gas-ccs:GRIDREGION-JPN-CB', 'gas-ccs:GRIDREGION-JPN-HK', 'gas-ccs:GRIDREGION-JPN-KA', 'gas-ccs:GRIDREGION-JPN-CG', 'gas-ccs:GRIDREGION-JPN-TK', 'gas-ccs:GRIDREGION-JPN-TH', 'gas-ccs:GRIDREGION-JPN-KY', 'coal-unspecified:GRIDREGION-JPN-SH', 'coal-unspecified:GRIDREGION-JPN-HR', 'coal-unspecified:GRIDREGION-JPN-CB', 'coal-unspecified:GRIDREGION-JPN-HK', 'coal-unspecified:GRIDREGION-JPN-KA', 'coal-unspecified:GRIDREGION-JPN-CG', 'coal-unspecified:GRIDREGION-JPN-TK', 'coal-unspecified:GRIDREGION-JPN-TH', 'coal-unspecified:GRIDREGION-JPN-KY', 'gas-unspecified:GRIDREGION-JPN-SH', 'gas-unspecified:GRIDREGION-JPN-HR', 'gas-unspecified:GRIDREGION-JPN-CB', 'gas-unspecified:GRIDREGION-JPN-HK', 'gas-unspecified:GRIDREGION-JPN-KA', 'gas-unspecified:GRIDREGION-JPN-CG', 'gas-unspecified:GRIDREGION-JPN-TK', 'gas-unspecified:GRIDREGION-JPN-TH', 'gas-unspecified:GRIDREGION-JPN-KY', 'coal-ammonia-cofiring:GRIDREGION-JPN-SH', 'coal-ammonia-cofiri

In [ ]:
n.optimize.solve_model(
    solver_name='highs',
    solver_options={
        'threads': 4, 
        'solver': "ipm",
        'run_crossover': "off",
        'small_matrix_value': 1e-5,
        'large_matrix_value': 1e9,
        'primal_feasibility_tolerance': 1e-4,
        'dual_feasibility_tolerance': 1e-4,
        'ipm_optimality_tolerance': 1e-4,
        'parallel': "on",
        'random_seed': 123,
        'log_to_console': True,
        'log_file': 'highs.log',
        'user_bound_scale': -2,
    }
)

INFO:linopy.model: Solve problem using Highs solver
INFO:linopy.model:Solver options:
 - threads: 4
 - solver: ipm
 - run_crossover: off
 - small_matrix_value: 1e-05
 - large_matrix_value: 1000000000.0
 - primal_feasibility_tolerance: 0.0001
 - dual_feasibility_tolerance: 0.0001
 - ipm_optimality_tolerance: 0.0001
 - parallel: on
 - random_seed: 123
 - log_to_console: True
 - log_file: highs.log
 - user_bound_scale: -2
INFO:linopy.io:Writing objective.
Writing continuous variables.: 100%|██████████| 8/8 [00:01<00:00,  6.45it/s]
INFO:linopy.io: Writing time: 14.97s


Running HiGHS 1.9.0 (git hash: fa40bdf): Copyright (c) 2024 HiGHS under MIT licence terms
Coefficient ranges:
  Matrix [1e-02, 6e+03]
  Cost   [1e+00, 5e+05]
  Bound  [5e+09, 5e+09]
  RHS    [1e+00, 2e+07]
Presolving model
1890317 rows, 1355339 cols, 8161583 nonzeros  2s
1601241 rows, 1329059 cols, 7329443 nonzeros  8s
Presolve : Reductions: rows 1601241(-2805171); columns 1329059(-282854); elements 7329443(-4126016)
Solving the presolved LP
IPX model has 1601241 rows, 1565575 columns and 7565959 nonzeros
Input
    Number of variables:                                1565575
    Number of free variables:                           0
    Number of constraints:                              1601241
    Number of equality constraints:                     473036
    Number of matrix entries:                           7565959
    Matrix range:                                       [1e-02, 6e+03]
    RHS range:                                          [1e+02, 2e+07]
    Objective range:        

In [ ]:
n.statistics(
)

In [ ]:
n.export_to_netcdf("/home/jy/tz-amp/.amp/client-earth_CE-A-SEP7-005_v3-dispatch-sen-nuc-testing-beta/platform_network.solved.nc")